# Text Summarisation

This notebook covers Text Summarisation, the task of creating a concise and fluent summary of a longer text document. Text summarisation can be either extractive (selecting important sentences from the original text) or abstractive (generating new sentences that capture the essence of the text).

We will explore both approaches using transformer-based models.

In [ ]:
# Import required libraries
from transformers import pipeline, SummarizationPipeline
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 1. Understanding Text Summarisation

Text summarisation has two main approaches:

**Extractive Summarisation**: 
- Selects the most important sentences directly from the original text
- Does not generate new text, just extracts existing sentences
- Simpler and more reliable, but may not flow as naturally

**Abstractive Summarisation**:
- Generates new sentences that convey the same meaning
- More like how humans summarise text
- Can produce more fluent summaries, but may introduce errors

Modern transformer-based models excel at abstractive summarisation.

In [ ]:
# Load pre-trained summarisation model
summariser = pipeline(
    "summarization",
    model="t5-small",
    tokenizer="t5-small"
)

print("=== Summarisation Model Loaded ===")
print("Model: t5-small")
print("T5 (Text-to-Text Transfer Transformer) is a text-to-text model.")

## 2. Basic Summarisation

Let's start with a simple example of text summarisation.

In [ ]:
# Sample article for summarisation
article = """
Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of "intelligent agents": any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals. 

Some high-profile applications of AI include advanced web search engines (e.g., Google), recommendation systems (YouTube, Amazon and Netflix), interacting via human-like voice assistants (Siri and Alexa), autonomous vehicles (Tesla), generative AI tools (ChatGPT and AI art), and superhuman play in strategy games (Chess and Go).

As machines become increasingly capable, tasks considered to require "intelligence" are often removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is frequently excluded from things considered to be AI, as it has become a routine technology.

The field was founded on the assumption that human intelligence "can be so precisely described that a machine can be made to simulate it". This raises philosophical arguments about the mind and the ethics of creating artificial beings designed to act like humans, which have been long explored in myth, fiction, and philosophy. 

Since its founding, AI has been subject to criticism and skepticism from scientists and the public. The term "AI" can also be used to describe machines that mimic cognitive functions that humans associate with the human mind, such as "learning" and "problem-solving".
"""

print("=== Original Article ===")
print(f"Length: {len(article)} characters")
print(f"Word count: {len(article.split())} words")
print("\nArticle:")
print(article)

In [ ]:
# Generate summary
summary = summariser(article, max_length=100, min_length=30, do_sample=False)

print("=== Generated Summary ===")
print(f"Length: {len(summary[0]['summary_text'])} characters")
print(f"Word count: {len(summary[0]['summary_text'].split())} words")
print("\nSummary:")
print(summary[0]['summary_text'])

## 3. Different Summarisation Models

Let's explore different pre-trained summarisation models available in the HuggingFace hub.

In [ ]:
# Test with different models
test_article = """
Climate change is one of the most pressing issues facing our planet today. The Earth's average surface temperature has risen about 1.1 degrees Celsius since the late 19th century, a change driven largely by increased carbon dioxide emissions into the atmosphere. 

This warming is causing widespread changes in weather patterns, leading to more extreme weather events such as hurricanes, droughts, and floods. Sea levels are rising as glaciers melt and ocean water expands, threatening coastal communities around the world.

Scientists warn that without significant action to reduce greenhouse gas emissions, these trends will accelerate, leading to even more severe impacts on ecosystems, human health, and economies. Many countries are now implementing policies to transition to cleaner energy sources and reduce their carbon footprint.
"""

print("=== Testing Different Models ===\n")

# Model 1: T5-small
print("1. T5-small:")
t5_summariser = pipeline("summarization", model="t5-small")
result = t5_summariser(test_article, max_length=80, min_length=20)
print(f"   {result[0]['summary_text']}\n")

# Model 2: BART (facebook/bart-large-cnn)
print("2. BART (facebook/bart-large-cnn):")
bart_summariser = pipeline("summarization", model="facebook/bart-large-cnn")
result = bart_summariser(test_article, max_length=80, min_length=20)
print(f"   {result[0]['summary_text']}\n")

# Model 3: PEGASUS (google/pegasus-xsum)
print("3. PEGASUS (google/pegasus-xsum):")
pegasus_summariser = pipeline("summarization", model="google/pegasus-xsum")
result = pegasus_summariser(test_article, max_length=80, min_length=20)
print(f"   {result[0]['summary_text']}")

## 4. Extractive Summarisation

Extractive summarisation selects the most important sentences from the original text. Let's implement a simple extractive summariser using sentence importance scores.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def extractive_summarise(text, num_sentences=3):
    """
    Simple extractive summarisation using TF-IDF and sentence similarity.
    """
    # Split text into sentences
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if len(sentences) <= num_sentences:
        return text
    
    # Create TF-IDF vectors for each sentence
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    
    # Calculate similarity of each sentence to all others
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    # Score each sentence by its similarity to all other sentences
    sentence_scores = similarity_matrix.sum(axis=1)
    
    # Get top N sentences
    top_indices = sentence_scores.argsort()[-num_sentences:][::-1]
    top_indices = sorted(top_indices)  # Keep original order
    
    # Create summary
    summary = ' '.join([sentences[i] for i in top_indices])
    
    return summary

# Test extractive summarisation
test_text = """
The quick brown fox jumps over the lazy dog. This is a well-known pangram that contains every letter of the English alphabet. 

Foxes are wild canids that are found throughout the world. They are known for their cunning and adaptability. 

Dogs are domesticated mammals that have been companions to humans for thousands of years. They come in many different breeds.

The brown fox is a common colour variation of the red fox. Foxes typically eat small mammals, birds, and insects.

Laziness is a trait associated with inactivity or reluctance to work. The term is often used informally.
"""

print("=== Extractive Summarisation ===\n")
print("Original text:")
print(test_text)
print("\n" + "="*50)

summary = extractive_summarise(test_text, num_sentences=2)
print("\nExtracted Summary (2 sentences):")
print(summary)

## 5. Summarising Different Types of Content

Let's test summarisation on different types of content.

In [ ]:
# Test on different content types
content_samples = {
    "News Article": """
Technology giant Apple announced today that it will launch a new line of smartphones next month. The new devices will feature advanced artificial intelligence capabilities and improved battery life. 

The company also revealed plans to expand its services business, including a new streaming service and fitness platform. Industry analysts predict these moves will help Apple compete more effectively with rivals in the technology sector.

Shares of Apple rose 2% in early trading following the announcement. The company's market capitalisation now exceeds $2 trillion, making it one of the most valuable companies in the world.
""",
    
    "Scientific Abstract": """
This study investigates the effects of climate change on global agricultural productivity. Using satellite data and crop yield records from 1980 to 2020, we analyse the impact of rising temperatures and changing precipitation patterns on major crops including wheat, rice, and maize.

Our findings indicate that climate change has reduced global agricultural yields by approximately 5% over the past four decades. However, the impact varies significantly by region, with tropical areas showing greater vulnerability compared to temperate regions.

We propose adaptation strategies including crop diversification, improved irrigation systems, and development of heat-resistant crop varieties to mitigate future impacts on food security.
""",
    
    "Meeting Notes": """
The team met to discuss the quarterly results. Sales exceeded targets by 15%, driven by strong performance in the European market. The marketing team presented new campaign ideas for the upcoming holiday season. 

The product development team reported progress on the new feature release, scheduled for next month. Several team members raised concerns about the timeline, suggesting additional resources may be needed.

The meeting concluded with action items assigned for each team. The next review is scheduled for two weeks from today.
"""
}

print("=== Summarising Different Content Types ===\n")

for content_type, content in content_samples.items():
    print(f"\n{'='*50}")
print(f"Content Type: {content_type}")
print(f"{'='*50}")
    
    summary = summariser(content, max_length=60, min_length=20)
    print(f"\nOriginal length: {len(content.split())} words")
    print(f"Summary length: {len(summary[0]['summary_text'].split())} words")
    print(f"\nSummary: {summary[0]['summary_text']}")

## 6. Fine-tuning Summarisation Models

For domain-specific summarisation, you may need to fine-tune a model on your specific data.

In [ ]:
print("=== Fine-tuning Summarisation Models ===")
print("""
To fine-tune a summarisation model:

```python
from transformers import Trainer, TrainingArguments
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Prepare dataset
def preprocess_function(examples):
    inputs = tokenizer(examples['source'], max_length=1024, truncation=True)
    labels = tokenizer(examples['target'], max_length=128, truncation=True)
    inputs['labels'] = labels['input_ids']
    return inputs

# Fine-tune
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
)
trainer.train()
```

Note: Fine-tuning requires a dataset with source-target pairs.
""")

## Summary

In this notebook, we covered Text Summarisation:

1. **Understanding**: Learned about extractive vs abstractive summarisation
2. **T5 Model**: Used T5 for text-to-text summarisation
3. **Different Models**: Compared T5, BART, and PEGASUS
4. **Extractive**: Implemented simple extractive summarisation
5. **Content Types**: Tested summarisation on various content types
6. **Fine-tuning**: Learned about fine-tuning for domain-specific summarisation

Text summarisation is a powerful tool for processing large volumes of text. The choice between extractive and abstractive depends on your specific use case and quality requirements.